In [13]:
# RNNによる文章生成
# 言語モデルを使った文章生成
# RNNによる文章生成の手順

In [14]:
# RnnlmGenクラスの実装
import sys
sys.path.append("..")
import numpy as np
from common.functions import softmax
from ch06.rnnlm import Rnnlm
from ch06.better_rnnlm import BetterRnnlm

class RnnlmGen(Rnnlm):
    def generate(self, start_id, skip_ids=None, sample_size=100):
        word_ids = [start_id]

        x = start_id
        while len(word_ids) < sample_size:
            x = np.array(x).reshape(1, 1)
            score = self.predict(x)
            p = softmax(score.flatten())

            sampled = np.random.choice(len(p), size=1, p=p)
            if (skip_ids is None) or (sampled not in skip_ids):
                x = sampled
                word_ids.append(int(x))

        return word_ids

In [15]:
# 文章生成する
import sys
sys.path.append("..")
from ch07.generate_text import word_ids
from rnnlm_gen import RnnlmGen
from dataset import ptb

corpus, word_to_id, id_to_word = ptb.load_data("train")
vocab_size = len(word_to_id)
corpus_size = len(corpus)

model = RnnlmGen()

# start文字とskip文字の設定
start_word = "you"
start_id = word_to_id[start_word]
skip_words = ["N", "<unk>", "s"]
skip_ids = [word_to_id[w] for w in skip_words]

# 文章生成
word_ids = model.generate(start_id, skip_ids)
txt = " ".join([id_to_word[i] for i in word_ids])
txt = txt.replace(" <eos> ", ".\n")
print(txt)

you substance defer lion birthday enviropact restrictions styles breakdown proven bounce concept essential intervention starts dairy shelves lawn lined backer richfield barbara moss mehta enforcers eric nothing distributed plant butcher lines international cardiovascular friendship my termination 2-for-1 precise per-share remove discuss fried deduction photographs share redevelopment vehicle terminate mountain us$ experiencing proves cosmetics representatives fruit vietnam enhance robin ounces seagate seldom athletic conform korea booklets ceo judge ratified always texaco neighbors proportion allocated met mtm long esselte trend durable poured mac ltd. sent minus shocks mentioned instance projected host slated anti-takeover implied closings closing public-relations clue sustain soap various fashionable


In [16]:
# seq2seqの原理
# 時系列データ変換用のトイ・プロブレム
# 可変長の時系列データ

In [19]:
# 足し算データセット
import sys 
sys.path.append("..")
from dataset import sequence

(x_train, t_train), (x_test, t_test) = \
    sequence.load_data("addition.txt", seed=1984)
char_to_id, id_to_char = sequence.get_vocab()

print(x_train.shape, t_train.shape)
print(x_test.shape, t_test.shape)
print(" ")
print(x_train[0])
print(t_train[0])
print(" ")
print("".join([id_to_char[c] for c in x_train[0]]))
print("".join([id_to_char[c] for c in t_train[0]]))


(45000, 7) (45000, 5)
(5000, 7) (5000, 5)
 
[ 3  0  2  0  0 11  5]
[ 6  0 11  7  5]
 
71+118 
_189 


In [21]:
# seq2seqの実装
# Encoderクラス
from common.time_layers import TimeLSTM
from common.time_layers import TimeEmbedding


rng = np.random.default_rng()

class Encoder:
    def __init__(self, vocab_size, wordvec_size, hidden_size):
        V, D, H = vocab_size, wordvec_size, hidden_size
        rn = rng.random()

        embed_W = (rn(V, D) / 100).astype("f")
        lstm_Wx = (rn(D, 4 * H) / np.sqrt(D)).astype("f")
        lstm_Wh = (rn(H, 4 * H) / np.sqrt(H)).astype("f")
        lstm_b = np.zeros(4 * H).astype("f")

        self.embed = TimeEmbedding(embed_W)
        self.lstm = TimeLSTM(lstm_Wx, lstm_Wh, lstm_b, stateful=False)

        self.params = self.embed.params + self.lstm.params
        self.grads = self.embed.grads + self.lstm.grads
        self.hs = None

    def forward(self, xs):
        xs = self.embed.forward(xs)
        hs = self.lstm.forward(xs)
        self.hs = hs
        return hs[:, -1, :]

    def backward(self, dh):
        dhs = np.zeros_like(self.hs)
        dhs[:, -1, :] = dh

        dout = self.lstm.backward(dhs)
        dout = self.embed.backward(dout)
        return dout